In [ ]:
import cv2
import numpy as np
import pyautogui
import json

: 

In [15]:
def find_frog_template(frame, template_path='frog_template.PNG'):
    """البحث عن الضفدع باستخدام صورة النموذج المقصوصة"""
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    template = cv2.imread(template_path, 0)
    
    if template is None:
        return None

    w, h = template.shape[::-1]
    res = cv2.matchTemplate(gray_frame, template, cv2.TM_CCOEFF_NORMED)
    
    # عتبة تطابق مرنة 0.5 لتجاوز فروقات الإضاءة
    threshold = 0.5
    _, max_val, _, max_loc = cv2.minMaxLoc(res)

    if max_val >= threshold:
        center_x = max_loc[0] + w // 2
        center_y = max_loc[1] + h // 2
        return (center_x, center_y)
    return None

In [16]:
def find_frog_by_shape(frame):
    """البحث عن الضفدع عبر كشف القاعدة الدائرية (Hough Circles)"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)

    # كشف الدوائر بمواصفات تناسب قاعدة الضفدع في زوما
    circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=1, minDist=100,
                               param1=50, param2=35, minRadius=40, maxRadius=100)

    if circles is not None:
        circles = np.uint16(np.around(circles))
        height, width = frame.shape[:2]
        screen_center = np.array([width/2, height/2])
        
        best_circle = None
        min_dist = float('inf')

        for i in circles[0, :]:
            # اختيار الدائرة الأقرب لمركز الشاشة لضمان أنه الضفدع وليس كرة 
            dist = np.linalg.norm(np.array([i[0], i[1]]) - screen_center)
            if dist < min_dist:
                min_dist = dist
                best_circle = (int(i[0]), int(i[1]))
        return best_circle
    return None

In [17]:
def save_frog_position(frog_pos, filename="frog_position.json"):
    data = {
        "x": int(frog_pos[0]),
        "y": int(frog_pos[1])
    }
    with open(filename, "w") as f:
        json.dump(data, f)


In [18]:
def run_zuma_bot():
    print("بدء التشغيل... يرجى الضغط على 'q' للخروج")
    
    while True:
        # قراءة الصورة (تأكد من وجود image.png في المجلد)
        frame = cv2.imread("image.png")
        
        if frame is None:
            print("خطأ: لم يتم العثور على صورة image.png")
            break

        # الخطوة 1: محاولة العثور بالنموذج (الطريقة الأولى)
        frog_pos = find_frog_template(frame)
        method_used = "Template"

        # الخطوة 2: إذا فشل، جرب كشف الدائرة (الطريقة الثانية)
        if frog_pos is None:
            frog_pos = find_frog_by_shape(frame)
            method_used = "Hough Circles"

        # رسم النتيجة وتوضيح الطريقة المستخدمة
        if frog_pos:
            save_frog_position(frog_pos)
            print("تم حفظ موقع الضفدع:", frog_pos)
            
            cv2.circle(frame, frog_pos, 20, (0, 255, 0), 2)
            cv2.putText(frame, f"Frog Found: {method_used}", (frog_pos[0]-50, frog_pos[1]-30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            print(f"تم تحديد الموقع: {frog_pos} باستخدام {method_used}")
        else:
            print("فشل العثور على الضفدع بكلا الطريقتين")

        cv2.imshow("Zuma Debug Mode", frame)
        
        # الانتظار حتى ضغط المفتاح (استخدم 0 للتوقف عند كل صورة أو 1 للزمن الحقيقي)
        if cv2.waitKey(0) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()

In [19]:
if __name__ == "__main__":
    run_zuma_bot()

بدء التشغيل... يرجى الضغط على 'q' للخروج
تم حفظ موقع الضفدع: (406, 348)
تم تحديد الموقع: (406, 348) باستخدام Hough Circles
